# Ноутбук со сравнением 3 датасетов

## 1. Импорт бибилиотек и конфигурация проекта

In [1]:
# mlflow ui --backend-store-uri sqlite:///C:/project/car-price-analyzer/src/mlflow_runs/mlflow.db

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import Ridge
from category_encoders.cat_boost import CatBoostEncoder
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GridSearchCV, KFold, RandomizedSearchCV
import copy
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_validate
import xgboost as xgb
import pyarrow
import phik
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import catboost
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate
)
from catboost import CatBoostRegressor
import mlflow.sklearn
from phik.report import plot_correlation_matrix
import plotly
import mlflow
import time
from scipy.stats import randint, uniform, loguniform
import os
from datetime import datetime
import category_encoders as ce
import joblib

c:\Users\Степан\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Создадим словарь конфигураций.

CONFIG = {
    # Константы
    "DEV_MODE": False,
    "DEV_SAMPLE_SIZE": 100000,
    "RANDOM_STATE": 42,
    # Целевая переменная 
    "TARGET": "Цена",
    "YEAR": datetime.now().year
}

In [4]:
df_raw = pd.read_parquet("../data/raw/df_optimal.parquet")

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    df_raw.drop(columns=[CONFIG["TARGET"]]), df_raw[CONFIG["TARGET"]], 
    test_size=0.2, 
    random_state=CONFIG["RANDOM_STATE"]
)

train_cleaned = pd.read_parquet("../data/cleaned/train_cleaned.parquet")
test_cleaned = pd.read_parquet("../data/cleaned/test_cleaned.parquet")

train_features = pd.read_parquet("../data/features/train_features.parquet")
test_features = pd.read_parquet("../data/features/test_features.parquet")

train_optimized = pd.read_parquet("../data/optimized/train_optimized.parquet")
test_optimized = pd.read_parquet("../data/optimized/test_optimized.parquet")

y_train_cleaned = train_cleaned[CONFIG["TARGET"]]
X_train_cleaned = train_cleaned.drop(columns=[CONFIG["TARGET"]])
y_test_cleaned = test_cleaned[CONFIG["TARGET"]]
X_test_cleaned = test_cleaned.drop(columns=[CONFIG["TARGET"]])

y_train_features = train_features[CONFIG["TARGET"]]
X_train_features = train_features.drop(columns=[CONFIG["TARGET"]])
y_test_features = test_features[CONFIG["TARGET"]]
X_test_features = test_features.drop(columns=[CONFIG["TARGET"]])

X_train_optimized = train_optimized.drop(columns=[CONFIG["TARGET"]])
y_train_optimized = train_optimized[CONFIG["TARGET"]]
X_test_optimized = test_optimized.drop(columns=[CONFIG["TARGET"]])
y_test_optimized = test_optimized[CONFIG["TARGET"]]

dir = 'C:/project/car-price-analyzer/src/mlflow_runs'
os.makedirs(dir, exist_ok=True)
mlflow.set_tracking_uri(f'sqlite:///{dir}/mlflow.db')
mlflow.set_experiment('dataset_compare')

<Experiment: artifact_location='file:c:/project/car-price-analyzer/researches/mlruns/4', creation_time=1785854392363, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1785854392363, lifecycle_stage='active', name='dataset_compare', tags={}, trace_location=None, workspace='default'>

In [5]:
scoring = {
    'mape': 'neg_mean_absolute_percentage_error',
    'mae': 'neg_mean_absolute_error'
}

## 2. Запуск эксперимента

In [6]:
datasets = {
    '1_raw_dataset': (X_train_raw, y_train_raw, X_test_raw, y_test_raw),
    '2_cleaned_dataset': (X_train_cleaned, y_train_cleaned, X_test_cleaned, y_test_cleaned),
    '3_features_dataset': (X_train_features, y_train_features, X_test_features, y_test_features),
    '4_optimized_dataset': (X_train_optimized, y_train_optimized, X_test_optimized, y_test_optimized)
}
cv = KFold(n_splits=5, shuffle=True, random_state=CONFIG["RANDOM_STATE"])
params = {
    'bootstrap_type': 'Bernoulli',
    'iterations': 2500,
    'learning_rate': 0.042248138023520114,
    'depth': 10,
    'l2_leaf_reg': 9.394802632887766,
    'random_strength': 0.8303775333704183,
    'border_count': 207,
    'subsample': 0.5483853226992782,
    'loss_function': 'MAE',
    'allow_writing_files': False,
    'eval_metric': 'MAE',
    'random_seed': 42,
    'thread_count': -1,
    'task_type': 'GPU',
    'verbose': 0
}

In [ ]:
for run_name, (X_train, y_train, X_test, y_test) in datasets.items():
    print(f"Запуск эксперимента для: {run_name}...")

    model = CatBoostRegressor(**params)
    wrapped_model = TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )

    X_train = X_train.copy()
    X_test = X_test.copy()
    y_train = y_train.copy()
    y_test = y_test.copy()

    text_features = ['Комплектация', 'Название машины']
    for col in text_features:
        X_train[col] = X_train[col].astype(str)
        X_test[col] = X_test[col].astype(str)

    cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in cat_features:
        X_train[col] = X_train[col].astype(object).fillna('unknown').astype(str)
        X_test[col] = X_test[col].astype(object).fillna('unknown').astype(str)

    with mlflow.start_run(run_name=run_name):

        scores = cross_validate(
            wrapped_model,
            X_train,
            y_train,
            cv=cv,
            params={'cat_features': cat_features, 'text_features': text_features},
            scoring=scoring
        )

        cv_mape = -scores['test_mape'].mean()
        cv_mae = -scores['test_mae'].mean()
        wrapped_model.fit(X_train, y_train, cat_features=cat_features, text_features=text_features)
        y_pred_test = wrapped_model.predict(X_test)

        test_mape = mean_absolute_percentage_error(y_test, y_pred_test)
        test_mae = mean_absolute_error(y_test, y_pred_test)

        # Логирование
        mlflow.log_metric('cv_mape', cv_mape)
        mlflow.log_metric('cv_mae', cv_mae)
        mlflow.log_metric('test_mape', test_mape)
        mlflow.log_metric('test_mae', test_mae)

        print(f"{run_name}:\nCV MAPE: {cv_mape:.4f} | CV MAE: {cv_mae:.0f} руб.\nTEST MAPE: {test_mape:.4f} | TEST MAE: {test_mae:.0f} руб.\n")

Запуск эксперимента для: 1_raw_dataset...


Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
Default metric period is 5 because MAE is/are not implemented for GPU
